[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/09_developability/09_developability_lab.ipynb)


# 09 — Developability — liability 모티프 직접 스캔

> 본문 [`09_developability.md`](09_developability.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행합니다.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나옵니다. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백합니다.
>
> **실측 소요 —** 정규식 liability 스캔(후보 5종) **1초 미만**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`09_developability`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `09_developability/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

이 셀이 끝나면 두 개의 경로가 준비됩니다 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "09_developability"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌립니다 — pip 만으로는 hmmscan 이 없습니다.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — liability 스캐너 구현 (본문 9.1~9.2)

**사람답지만 만들 수 없는 항체는 약이 못 됩니다.** 후보를 만들 때마다 모티프 스캔을 자동으로 돌리세요.

| 모티프 | 정규식 | 위험 |
|---|---|---|
| N-glycosylation | `N[^P][ST]` | 예상치 못한 당쇄 → 이질성·클리어런스 |
| deamidation | `N[GS]` | 보관 중 전하 변이 |
| isomerization | `DG` | 구조 변형 |
| oxidation | `[MW]` | 산화 → 활성 저하 |

핵심은 **parental 대비 증분** — humanization 이 **새로 만든** liability 를 잡는 것입니다.

In [ ]:
import re, pandas as pd

MOTIFS = {
    "N-glycosylation": r"N[^P][ST]",
    "deamidation":     r"N[GS]",
    "isomerization":   r"DG",
    "oxidation":       r"[MW]",
}

def scan(seq):
    return {name: [m.start() + 1 for m in re.finditer(p, seq)] for name, p in MOTIFS.items()}

# 후보 모으기 — 앞 랩의 my_run 우선
cands = {"parental": (VH, VL)}
for chapter, fname, keys, label in [
    ("05_humanize_sapiens", "sapiens_humanized_noguard.fasta", ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens"),
    ("06_cdr_safe_tools",   "humatch_humanised.fasta",         ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch"),
    ("06_cdr_safe_tools",   "anthroab_best_score.fasta",       ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab"),
    ("06_cdr_safe_tools",   "anthroab_masked_FRonly.fasta",    ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked"),
]:
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p); cands[label] = (f[keys[0]], f[keys[1]])
        print(f"[내 결과 · {chapter}] {p}")

if len(cands) == 1:
    print("[레퍼런스] data/variants.fasta")
    v = read_fasta("data/variants.fasta")
    for n in ["parental", "sapiens", "humatch", "anthroab", "anthroabFRmasked"]:
        cands[n] = (v[f"{n}_VH"], v[f"{n}_VL"])

rows = []
for name, (h, l) in cands.items():
    for chain, seq in (("VH", h), ("VL", l)):
        for motif, hits in scan(seq).items():
            rows.append({"candidate": name, "chain": chain, "motif": motif,
                         "count": len(hits), "positions_1based": ";".join(map(str, hits))})
lia = pd.DataFrame(rows)
lia.to_csv(MY / "liability.csv", index=False)
print("\n→", MY / "liability.csv")
display(lia.pivot_table(index="candidate", columns=["chain", "motif"], values="count", fill_value=0))

## 2) 내 결과 확인 — **parental 에 없던** 모티프만 (증분 스캔)

절대 개수보다 중요한 건 "humanization 이 **새로 만든** liability" 입니다.

In [ ]:
par_scan = {"VH": scan(cands["parental"][0]), "VL": scan(cands["parental"][1])}

new_rows = []
for name, (h, l) in cands.items():
    if name == "parental":
        continue
    for chain, seq in (("VH", h), ("VL", l)):
        for motif, hits in scan(seq).items():
            base = set(par_scan[chain][motif])
            new = sorted(set(hits) - base)
            lost = sorted(base - set(hits))
            new_rows.append({"candidate": name, "chain": chain, "motif": motif,
                             "신규": len(new), "신규 위치": ";".join(map(str, new)) or "-",
                             "사라짐": len(lost)})
delta = pd.DataFrame(new_rows)
delta.to_csv(MY / "liability_delta.csv", index=False)

glyc = delta[(delta.motif == "N-glycosylation") & (delta["신규"] > 0)]
print("신규 N-glycosylation 모티프(가장 위험):", "없음" if glyc.empty else "")
if not glyc.empty:
    display(glyc)
display(delta[delta["신규"] > 0].sort_values(["candidate", "chain"]))
print("\n읽는 법 — 신규 N-glyc 모티프가 CDR/paratope 근처에 생기면 hard filter 대상입니다(Ch.10).")

## 3) 그래프 — 후보별 liability 총량

In [ ]:
from humanization_viz import liability_overview

rows = (lia.groupby(["candidate", "motif"])["count"].sum().reset_index()
          .rename(columns={"count": "count"}).to_dict("records"))
liability_overview(rows, "Developability liability motifs (VH+VL)", "09_liability.png")
from IPython.display import Image; Image("09_liability.png")

## 4) CDR 안에 떨어진 liability 인가?

같은 모티프라도 **CDR/paratope 안**이면 위험도가 다릅니다(결합에 직접 영향).


In [ ]:
ct = pd.read_csv("data/cdr_table_imgt.csv")
guard = {"H": set(), "L": set()}
for _, r in ct.iterrows():
    seq = VH if r["chain"] == "H" else VL
    st = seq.find(r["sequence"]) + 1
    guard[r["chain"]] |= set(range(st, st + len(r["sequence"])))

hits = []
for name, (h, l) in cands.items():
    for chain, seq, tag in (("VH", h, "H"), ("VL", l, "L")):
        for motif, ps in scan(seq).items():
            inside = [p for p in ps if p in guard[tag]]
            if inside:
                hits.append({"candidate": name, "chain": chain, "motif": motif,
                             "CDR 안 위치": ";".join(map(str, inside))})
cdr_lia = pd.DataFrame(hits)
display(cdr_lia if not cdr_lia.empty else "CDR 안 liability 없음")
print("주의 — CDR 좌표는 parental 기준입니다. indel 이 있는 후보(Humatch VL)는 위치가 한 칸씩 밀릴 수 있습니다.")

## 5) 레퍼런스 대조

In [ ]:
ref = pd.read_csv("data/liability_reference.csv")
piv_ref = ref.pivot_table(index="candidate", columns="motif", values="count", aggfunc="sum", fill_value=0)
piv_my  = lia.pivot_table(index="candidate", columns="motif", values="count", aggfunc="sum", fill_value=0)
print("[레퍼런스]"); display(piv_ref)
common = piv_my.index.intersection(piv_ref.index)
print("내 결과와 레퍼런스가 일치:", piv_my.loc[common].equals(piv_ref.loc[common]))

## 이 랩에서 확인한 것

1. liability 스캔은 **정규식 4줄**이면 끝납니다 — 후보를 만들 때마다 자동으로 돌리세요.
2. 중요한 건 절대 개수가 아니라 **parental 대비 신규 모티프**입니다(특히 신규 `NXS/T`).
3. CDR 안에 떨어진 모티프는 위험도가 다릅니다 — Ch.10 의 **hard filter** 로 연결됩니다.
4. 이 표가 랭킹의 developability 항목이 됩니다.

다음 → **[Ch.10 — 랭킹·리포트](../10_ranking_report/10_ranking_lab.ipynb)**